In [1]:
import pandas as pd

In [2]:
df_cases = pd.read_csv("judicial_decisions_cleaned.csv")
df_vdem = pd.read_csv("VDEMv16SelectedVars_data.csv")
df_dict = pd.read_csv("VDEMv16SelectedVars_dictionary.csv")

In [3]:
df_cases.columns = df_cases.columns.str.strip().str.lower()
df_vdem.columns = df_vdem.columns.str.strip().str.lower()
df_dict.columns = df_dict.columns.str.strip().str.lower()

In [4]:
country_map = {
    "United States": "United States of America",
    "Myanmar": "Burma/Myanmar",
    "Gambia": "The Gambia",
    "Moldova, Republic of": "Moldova",
    "Iran, Islamic Republic of": "Iran",
    "Korea, Republic of": "South Korea",
    "Russian Federation": "Russia",
    "Republic of North Macedonia": "North Macedonia",
    "Syrian Arab Republic": "Syria",
    "Czech Republic": "Czechia",
    "Lao People's Democratic Republic": "Laos",
    "DRC": "Democratic Republic of the Congo",
    "Suecia": "Sweden",
    "Venezuela, Bolivarian Republic of": "Venezuela",
    "Palestine": "Palestine/West Bank",
    "Palestine, State of": "Palestine/West Bank",
    "Turkey": "T√ºrkiye"
}

In [5]:
df_cases["country_clean"] = df_cases["country"].replace(country_map).str.strip()

In [6]:
def check_merge(df_cases: pd.DataFrame, df_vdem: pd.DataFrame) -> pd.DataFrame:
    df_temp = df_cases.merge(
        df_vdem,
        left_on=["country_clean", "year"],
        right_on=["country_name", "year"],
        how="left"
    )
    rate = df_temp["v2x_polyarchy"].notna().mean()
    print(f"Match rate: {rate:.2%}")
    print("\nTop unmatched countries:")
    print(df_temp.loc[df_temp["v2x_polyarchy"].isna(), "country"].value_counts().head(20))
    return df_temp

In [7]:
df_merged = check_merge(df_cases, df_vdem)

Match rate: 98.73%

Top unmatched countries:
country
International                    19
Meta                              3
EU                                2
European Union                    2
Liechtenstein                     2
Bermuda                           1
Holy See (Vatican City State)     1
Puerto Rico                       1
Saint Lucia                       1
United States                     1
Name: count, dtype: int64


In [8]:
df_merged["vdem_matched"] = df_merged["v2x_polyarchy"].notna()

In [9]:
df_merged.to_csv("judicial_decisions_vdem_merged.csv", index=False)

In [10]:
vars_to_lag = [
    "v2x_polyarchy",
    "v2x_freexp_altinf",
    "v2x_freexp",
    "v2x_jucon",
    "v2jureform",
    "v2jupurge",
    "v2jupoatck",
    "v2jupack",
    "v2juaccnt",
    "v2jucorrdc",
    "v2juhcind",
    "v2juncind",
    "v2juhccomp",
    "v2jucomp",
    "v2jureview",
]

In [11]:
df_cases["country_clean"] = df_cases["country"].replace(country_map).str.strip()

In [12]:
lag_source = df_vdem[["country_name", "year"] + vars_to_lag].copy()
lag_source = lag_source.rename(columns={"country_name": "country_clean"})

In [13]:
lag_source["year"] = lag_source["year"].astype(int)
df_cases["year"] = df_cases["year"].astype(int)

In [14]:
df_final = df_merged.copy()

In [15]:
for lag in [1, 2]:
    temp = lag_source.copy()
    temp["year"] += lag
    temp = temp.rename(columns={v: f"{v}_lag{lag}" for v in vars_to_lag})

    df_final = df_final.merge(
        temp,
        on=["country_clean", "year"],
        how="left"
    )

In [16]:
df_final.to_csv("judicial_decisions_merged_with_lags.csv", index=False)